#### Imports

In [ ]:
# Imports
import os
import sys
import random
import json
from PIL import Image
from tqdm import tqdm
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as patches


#### Path Configurations

In [ ]:
# --- Project root (2 levels up from this notebook) ---
project_root = os.path.abspath(os.path.join('..', '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# --- Path configuration ---
resolution   = 800
dataset_name = "coco_football_players_detection_v11"

data_directory               = os.path.join(project_root, "data")
INPUT_ROOT                   = os.path.join(data_directory, "detection", dataset_name)
OUTPUT_ROOT                  = os.path.join(data_directory, "detection", f"{resolution}_{dataset_name}")

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# --- Splits & annotation filename ---
SPLITS              = ["train", "valid", "test"]  # remove any that don't exist
ANNOTATION_FILENAME = "_annotations.coco.json"

# --- Target size ---
TARGET_WIDTH  = resolution
TARGET_HEIGHT = resolution

# --- Resize mode ---
# 'fit'      : letterbox/pad to keep aspect ratio (recommended for DETR models)
# 'stretch'  : resize to exact size (may distort)
# 'crop'     : resize then center-crop
RESIZE_MODE = "fit"
PAD_COLOR   = (114, 114, 114)  # grey padding, used only in 'fit' mode

RESAMPLE = Image.LANCZOS

print(f"Input  : {INPUT_ROOT}")
print(f"Output : {OUTPUT_ROOT}")
print(f"Target : {TARGET_WIDTH}×{TARGET_HEIGHT} | mode: {RESIZE_MODE}")

#### Utility Imports

In [ ]:
from utils.data import process_split

#### Run all splits

In [ ]:
summary = {}

for split in SPLITS:
    print(f"\nProcessing '{split}'...")
    stats = process_split(
        split, INPUT_ROOT, OUTPUT_ROOT,
        TARGET_WIDTH, TARGET_HEIGHT,
        RESIZE_MODE, PAD_COLOR, RESAMPLE,
        ANNOTATION_FILENAME,
    )
    if stats:
        summary[split] = stats
        warn = f" | ⚠ {stats['missing']} missing" if stats['missing'] else ""
        print(f"  ✓ {stats['images']} images, {stats['annotations']} annotations{warn}")

print("\n" + "="*45)
print(f"{'Split':<10} {'Images':>8} {'Annotations':>14} {'Missing':>9}")
print("-"*45)
for split, s in summary.items():
    print(f"{split:<10} {s['images']:>8} {s['annotations']:>14} {s['missing']:>9}")
print("="*45)
print(f"Output → {OUTPUT_ROOT}/")

#### Sanity Check

In [ ]:
fig, axes = plt.subplots(1, len(summary), figsize=(6 * len(summary), 6))
if len(summary) == 1:
    axes = [axes]

for ax, (split, _) in zip(axes, summary.items()):
    ann_path = Path(OUTPUT_ROOT) / split / ANNOTATION_FILENAME
    img_dir  = Path(OUTPUT_ROOT) / split

    with open(ann_path) as f:
        coco = json.load(f)

    ann_img_ids = list({a["image_id"] for a in coco["annotations"]})
    if not ann_img_ids:
        ax.set_title(f"{split} (no annotations)")
        ax.axis("off")
        continue

    sample_id   = random.choice(ann_img_ids)
    sample_rec  = next(i for i in coco["images"] if i["id"] == sample_id)
    sample_anns = [a for a in coco["annotations"] if a["image_id"] == sample_id]

    img = Image.open(img_dir / sample_rec["file_name"])
    ax.imshow(img)
    for ann in sample_anns:
        if "bbox" in ann and ann["bbox"]:
            x, y, w, h = ann["bbox"]
            ax.add_patch(patches.Rectangle(
                (x, y), w, h, linewidth=1.5, edgecolor="lime", facecolor="none"))
    ax.set_title(f"{split} | {img.size[0]}×{img.size[1]} | {len(sample_anns)} anns")
    ax.axis("off")

plt.tight_layout()
plt.show()